In [ ]:
import cv2
import numpy as np
import os
import PIL
from PIL import Image
import pandas as pd
from shutil import copyfile
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import timeit
from datetime import datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Préparation des données

Combinaison des segments des images

In [ ]:
folder_to_load = "Mask-segments"
folder_to_save = "Mask-segments_combined"

segments_labels = ['skin', 'nose', 'eye_g', 'l_eye', 'r_eye', 'l_brow',
                   'r_brow', 'l_ear', 'r_ear', 'mouth', 'u_lip', 'l_lip',
                   'hair', 'hat', 'ear_r', 'neck_l', 'neck', 'cloth']
images_dataset_size = 30000

for i in range(images_dataset_size):

    folder = i // images_dataset_size
    base_image = np.zeros((512, 512))

    for label_index, label in enumerate(segments_labels):
        filename = os.path.join(folder_to_load, str(folder), str(i).rjust(5, "0") + "_" + label + ".png")

        if (os.path.exists(filename)):
            im = cv2.imread(filename)
            im = im[:, :, 0]
            base_image[im != 0] = (label_index + 1)

    image_combined = os.path.join(folder_to_save, str(i) + ".png")
    cv2.imwrite(image_combined, base_image)

Coloration des segments selon la nature d'organe (yeux, bouche, etc..)

In [ ]:
folder_to_load = "Mask-segments_combined"
folder_to_save = "Mask-segments_colored"

segments_colors = [[0, 0, 0],
              [204, 0, 0],
              [76, 153, 0],
              [204, 204, 0],
              [51, 51, 255],
              [204, 0, 204],
              [0, 255, 255],
              [255, 204, 204],
              [102, 51, 0],
              [255, 0, 0],
              [102, 204, 0],
              [255, 255, 0],
              [0, 0, 153],
              [0, 0, 204],
              [255, 51, 153],
              [0, 204, 204],
              [0, 51, 0],
              [255, 153, 51],
              [0, 204, 0]]

for i in range(images_dataset_size):
    base_image = np.zeros((512, 512, 3))
    filename = os.path.join(folder_to_load, str(i) + ".png")

    if (os.path.exists(filename)):
        image_segments = Image.open(filename)
        image_segments = np.array(image_segments)

        for label_index, label_color in enumerate(segments_colors):
            label_value = label_index + 1

            base_image[image_segments == label_value] = label_color

        image_colored = os.path.join(folder_to_save, str(i) + ".png")
        cv2.imwrite(image_colored, base_image)

In [ ]:
#Rendre les images d'annotation en niveau de gris
folder_to_load = "Mask-segments_combined"
folder_to_save = "Mask-segments_combined 2D"

for i in range(images_dataset_size):
    filename = os.path.join(folder_to_load, str(i) + ".png")

    image = cv2.imread(filename)
    image_2D = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    new_image_path = os.path.join(folder_to_save, str(i) + ".png")
    cv2.imwrite(new_image_path, image_2D)

In [ ]:
import zipfile
#"train_labels.zip", "train_images.zip", "Model.zip", "Mask-segments_combined 2D.zip"
zip_files = ["test_labels.zip", "test_images.zip", "val_labels.zip", "val_images.zip", "train_labels.zip", "Model.zip", "Mask-segments_combined 2D.zip"]

for zip_file in zip_files:
  with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall()

In [ ]:
import zipfile
Drive_PATH = "drive/MyDrive"
zip_files = ["train_images.zip"]

for zip_file in zip_files:
  zip_path = os.path.join(Drive_PATH, zip_file)
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall()

Séparation et organisation des images

In [ ]:
#Vider les répertoires de Train, Validation et Test
folders = ["train_labels", "train_images", "test_labels", "test_images", "val_labels", "val_images"]

for folder in folders:
    folder_path = os.path.join(os.getcwd() ,folder)
    for filename in os.listdir(folder_path):
        if os.path.isfile(os.path.join(folder_path, filename)):
            os.remove(os.path.join(folder_path, filename))

In [ ]:
data_label = "Mask-segments_combined 2D"
data_image = "data_images"
train_label = "train_labels"
train_image = "train_images"
val_label = "val_labels"
val_image = "val_images"
test_label = "test_labels"
test_image = "test_images"

In [ ]:
images = pd.read_csv("images-mapping.txt", delim_whitespace=True, header=None, skiprows=[0])

#train_data = open("train.txt", "w")
#train_data.write("index        image_id   val_id\n")
#train_size = 0

#val_data = open("val.txt", "w")
#val_data.write("index        image_id  val_id\n")
#val_size = 0

#test_data = open("test.txt", "w")
#test_data.write("index        image_id   test_id\n")
#test_size = 0

for index, image_id in enumerate(images.iloc[:, 1]):

    if index >= 0 and index <= 1: #image_id >= 162771 and image_id <= 182638:
        copyfile(os.path.join(data_label, str(index) + ".png"), os.path.join(val_label, str(val_size) + ".png"))
        copyfile(os.path.join(data_image, str(index) + ".jpg"), os.path.join(val_image, str(val_size) + ".jpg"))
        val_data.write(str(index) + "        " + str(image_id) + "   " + str(val_size) + "\n")
        val_size += 1

    elif index >=2 and index <=4: #image_id >= 182638:
        copyfile(os.path.join(data_label, str(index) + ".png"), os.path.join(test_label, str(test_size) + ".png"))
        copyfile(os.path.join(data_image, str(index) + ".jpg"), os.path.join(test_image, str(test_size) + ".jpg"))
        test_data.write(str(index) + "        " + str(image_id) + "   " + str(test_size) + "\n")
        test_size += 1

    elif index >= 5 and index <= 14:
        copyfile(os.path.join(data_label, str(index) + ".png"), os.path.join(train_label, str(train_size) + ".png"))
        copyfile(os.path.join(data_image, str(index) + ".jpg"), os.path.join(train_image, str(train_size) + ".jpg"))
        train_data.write(str(index) + "        " + str(image_id) + "   " + str(train_size) + "\n")
        train_size += 1

train_data.close()
val_data.close()
test_data.close()

Préparation de l'ensemble d'entrainement et l'ensemble du test

In [ ]:
train_dataset = []
test_dataset = []

train_df = pd.read_csv("train.txt", delim_whitespace=True, header=None, skiprows=[0])
test_df = pd.read_csv("test.txt", delim_whitespace=True, header=None, skiprows=[0])

for image_id in train_df.loc[:, 2]:
    train_image_path = os.path.join(os.getcwd(), train_image, str(image_id) + ".jpg")
    train_label_path = os.path.join(os.getcwd(), train_label, str(image_id) + ".png")

    train_dataset.append([train_image_path, train_label_path])

for image_id in test_df.loc[:, 2]:
    test_image_path = os.path.join(os.getcwd(), test_image, str(image_id) + ".jpg")
    test_label_path = os.path.join(os.getcwd(), test_label, str(image_id) + ".png")

    test_dataset.append([test_image_path, test_label_path])

Chargement des données

In [ ]:
#transformer_trainer --> transformer_label
#transformer_tester --> transformer_img

class dataset():
    def __init__(self, training_mode):
        self.training_mode = training_mode
        self.dataset = train_dataset if self.training_mode == True else test_dataset
        self.dataset_size = len(train_dataset) if self.training_mode == True else len(test_dataset)

        self.transformer_label = transforms.ToTensor()
        self.transformer_img = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __getitem__(self, index):
        image_path, label_path = self.dataset[index]
        image = Image.open(image_path)
        label = Image.open(label_path)

        if self.training_mode:
            return self.transformer_img(image), self.transformer_label(label)
        else:
            return self.transformer_img(image), self.transformer_label(label)

    def __len__(self):
        return self.dataset_size

In [ ]:
#MODIFIED num_workers FROM 0 TO 2
#MODIFIED batch_size FROM 2 TO 10

class data_loader():
    def __init__(self, dataset):
        self.dataset = dataset

    def loading(self, shuffle):
        load = torch.utils.data.DataLoader(dataset=self.dataset,
        batch_size=10, shuffle=shuffle, num_workers=2, drop_last=False)

        return load

Construction du modéle

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class convolution(nn.Module):
    def __init__(self, input_size, output_size):
        super(convolution, self).__init__()

        self.convol1 = nn.Sequential(
            nn.Conv2d(input_size, output_size, 3, 1, 1),
            nn.BatchNorm2d(output_size),
            nn.ReLU()
            )

        self.convol2 = nn.Sequential(
            nn.Conv2d(output_size, output_size, 3, 1, 1),
            nn.BatchNorm2d(output_size),
            nn.ReLU()
            )

    def forward(self, input):
        output = self.convol1(input)
        output = self.convol2(output)

        return output

In [ ]:
class deconvolution(nn.Module):
    def __init__(self, input_size, output_size):
        super(deconvolution, self).__init__()

        self.conv = convolution(input_size, output_size)
        self.deconv = nn.ConvTranspose2d(input_size, output_size, kernel_size=2, stride=2)

    def forward(self, input1, input2):
        output2 = self.deconv(input2)

        offset = output2.size()[2] - input1.size()[2]
        padding = 2 * [offset // 2, offset // 2]

        output1 = F.pad(input1, padding)

        output_final = self.conv(torch.cat([output1, output2], 1))

        return output_final

In [ ]:
#MODIFIED filters TO int(filters/4)

class model(nn.Module):
    def __init__(self, n_classes=19):
        super(model, self).__init__()

        filters = [64, 128, 256, 512, 1024]
        filters = [int(x / 4) for x in filters]

        self.conv1 = convolution(3, filters[0])
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)

        self.conv2 = convolution(filters[0], filters[1])
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)

        self.conv3 = convolution(filters[1], filters[2])
        self.maxpool3 = nn.MaxPool2d(kernel_size=2)

        self.conv4 = convolution(filters[2], filters[3])
        #self.maxpool4 = nn.MaxPool2d(kernel_size=2)

        #self.conv5 = convolution(filters[3], filters[4])

        #self.deconv4 = deconvolution(filters[4], filters[3])
        self.deconv3 = deconvolution(filters[3], filters[2])
        self.deconv2 = deconvolution(filters[2], filters[1])
        self.deconv1 = deconvolution(filters[1], filters[0])

        self.convfinal = nn.Conv2d(filters[0], n_classes, 1)

    def forward(self, input):
        conv1 = self.conv1(input)
        maxpool1 = self.maxpool1(conv1)

        conv2 = self.conv2(maxpool1)
        maxpool2 = self.maxpool2(conv2)

        conv3 = self.conv3(maxpool2)
        maxpool3 = self.maxpool3(conv3)

        conv4 = self.conv4(maxpool3)
        #maxpool4 = self.maxpool4(conv4)

        #conv5 = self.conv5(maxpool4)

        #deconv4 = self.deconv4(conv4, conv5)
        deconv3 = self.deconv3(conv3, conv4)
        deconv2 = self.deconv2(conv2, deconv3)
        deconv1 = self.deconv1(conv1, deconv2)

        convfinal = self.convfinal(deconv1)

        return convfinal


In [ ]:
class training():
    def __init__(self, data_load):
        self.data_load = data_load
        self.total_steps = len(data_load)
        self.batch_size = data_load.batch_size
        self.epochs = 50

        image_size = 512
        self.labels_transformer = transforms.Resize((image_size, image_size), interpolation = PIL.Image.NEAREST)

        self.building_model()

    def building_model(self):
        self.model = model().to(device)

        self.criteria = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters())

    def fit(self):
        Data_iterator = iter(self.data_load)

        for step in range(self.total_steps * self.epochs):

            self.model.train()

            try:
              imgs, labels = next(Data_iterator)
            except:
              Data_iterator = iter(self.data_load)
              imgs, labels = next(Data_iterator)

            labels[:, 0, :, :] = labels[:, 0, :, :] * 255
            labels_real = labels[:, 0, :, :].cuda()

            imgs = imgs.cuda()
            predictions = self.model(imgs)
            predictions = self.labels_transformer(predictions)

            loss = self.criteria(predictions, labels_real.long())
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        self.save_model()

        print("training finished")

    def save_model(self):
        model_save_path = "Model"
        model_name = "Model.pth"

        if not os.path.isdir(model_save_path):
            os.makedirs(model_save_path)

        torch.save(self.model.state_dict(), os.path.join(os.getcwd(), model_save_path, model_name))

In [ ]:
class testing():
    def __init__(self, data_load):
        self.data_load = data_load
        self.total_steps = len(self.data_load)
        self.batch_size = data_load.batch_size
        self.model_save_path = "Model"
        self.model_name = "Model.pth"
        self.predictions_save_path = "test_predicted_labels"

        image_size = 512
        self.labels_transformer = transforms.Resize((image_size, image_size), interpolation = PIL.Image.NEAREST)

        self.building_model()

    def building_model(self):
        self.model = model().to(device)

    def test(self):
        self.model.load_state_dict(torch.load(os.path.join(os.getcwd(), self.model_save_path, self.model_name)))
        self.model.eval()

        Data_iterator = iter(self.data_load)

        for step in range(self.total_steps):
          images, labels = next(Data_iterator)

          #images_list = []

          #for image in images:
          #    images_list.append(image)

          #images_stack = torch.stack(images_list)
          #images_stack = images_stack.cuda()

          images = images.cuda()
          predictions = self.model(images)
          predictions = self.labels_transformer(predictions)

          #predictions_labels = self.generate_labels(predictions)

          predictions_list = []
          image_size = 512

          for image in predictions:
            image = image.view(1, 19, image_size, image_size)
            prediction = np.squeeze(image.data.max(1)[1].cpu().numpy(), axis=0)
            predictions_list.append(prediction)


          predictions_list = np.array(predictions_list)
          predictions_list = torch.from_numpy(predictions_list)

          predictions_labels = []

          for predicted_image in predictions_list:
              predictions_labels.append(predicted_image.numpy())

          predictions_labels = np.array(predictions_labels)

          if not os.path.isdir(self.predictions_save_path):
                os.makedirs(self.predictions_save_path)

          for index, prediction in enumerate(predictions_labels):
            cv2.imwrite(os.path.join(os.getcwd(), self.predictions_save_path, "{}.png".format(step * self.batch_size + index)), prediction)

    def generate_labels(prediction_images, image_size=512):
      predictions_list = []

      for image in prediction_images:
        image = image.view(1, 19, image_size, image_size)
        prediction = np.squeeze(image.data.max(1)[1].cuda().numpy(), axis=0)
        predictions_list.append(prediction)


      predictions_list = np.array(predictions_list)
      predictions_list = torch.from_numpy(predictions_list)

      labels = []

      for predicted_image in predictions_list:
          labels.append(predicted_image.numpy())

      labels = np.array(labels)

      return labels

In [ ]:
!nvidia-smi

Mon Jun  3 10:20:50 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   37C    P8               9W /  70W |      3MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
data_loads = data_loader(dataset(True)).loading(True)
trainer = training(data_load=data_loads)

start_time = datetime.now()
trainer.fit()
end_time = datetime.now()
duration = end_time - start_time

print("Duration : {}".format(duration))
#Data size - duration : 1000 image => 1 minutes
#Data size - duration : 30000 image => 4 hours 30 minutes

/usr/local/lib/python3.10/dist-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with an OutOfMemoryError: CUDA out of memory. Tried to allocate 4.22 GiB. GPU  (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:924.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


OutOfMemoryError: CUDA out of memory. Tried to allocate 640.00 MiB. GPU 

In [ ]:
data_loads = data_loader(dataset(False)).loading(False)

tester = testing(data_load=data_loads)
tester.test()

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
